In [1]:
# ============================================================
# STAGE 2 — Cell 1: Load Stage 0 artifacts directly
# ============================================================
!pip install huggingface_hub pyarrow scikit-learn scipy -q

import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow.parquet as pq
import pickle, json, re, gc, os
from pathlib import Path
from huggingface_hub import snapshot_download

# Constants
REPO_ID    = "datdong2004/amazonNew-cleaned"
CATEGORY   = "home_&_kitchen"
COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

STAGE0_DIR = Path("/kaggle/input/datasets/datdong123/home-and-kitchen-stage-0/data/stage0_artifacts")

with open(STAGE0_DIR / "split_meta.json") as f:
    meta = json.load(f)

global_mean = float(meta["global_mean"])
split_date  = pd.Timestamp(meta["split_date"])
m_user      = float(meta["m_user"])
m_item      = float(meta["m_item"])

train = pd.read_parquet(STAGE0_DIR / "train.parquet")
test  = pd.read_parquet(STAGE0_DIR / "test.parquet")
df_core = pd.read_parquet(STAGE0_DIR / "df_core.parquet")

train[COL_DATE] = pd.to_datetime(train[COL_DATE])
test[COL_DATE]  = pd.to_datetime(test[COL_DATE])
df_core[COL_DATE] = pd.to_datetime(df_core[COL_DATE])

print("=== Loaded from Stage 0 ===")
print(json.dumps(meta, indent=2))
print(f"Train   : {len(train):,}")
print(f"Test    : {len(test):,}")
print(f"df_core : {len(df_core):,}")

# Raw parquet cache chỉ để lấy text/meta columns
LOCAL_DIR = f"/kaggle/working/data/{CATEGORY}"
Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)

if not any(Path(LOCAL_DIR).rglob("*.parquet")):
    snapshot_download(
        repo_id        = REPO_ID,
        repo_type      = "dataset",
        local_dir      = LOCAL_DIR,
        allow_patterns = ["*home_&_kitchen*"],
        ignore_patterns= ["*.json", "*.md"],
    )

downloaded = []
for root, dirs, fnames in os.walk(LOCAL_DIR):
    for f in fnames:
        if f.endswith(".parquet"):
            full = Path(root) / f
            downloaded.append((str(full), full.stat().st_size / 1e6))

downloaded = sorted(downloaded, key=lambda x: x[0])

print(f"Files: {len(downloaded)}")
print(f"Total size: {sum(s for _, s in downloaded):.1f} MB")

train_pairs = set(zip(train[COL_USER], train[COL_ITEM]))
core_pairs  = set(zip(df_core[COL_USER], df_core[COL_ITEM]))

print(f"Train pairs: {len(train_pairs):,}")
print(f"Core pairs : {len(core_pairs):,}")

import psutil
ram = psutil.virtual_memory()
print(f"RAM available: {ram.available/1e9:.1f} GB")

=== Loaded from Stage 0 ===
{
  "category": "home_&_kitchen",
  "split_date": "2017-05-15 00:00:00",
  "split_idx": 4498249,
  "n_train": 4498249,
  "n_test": 1124563,
  "global_mean": 4.34061876076669,
  "global_std": 1.1346049162038023,
  "m_user": 6.0,
  "m_item": 9.0,
  "k_core": 5
}
Train   : 4,498,249
Test    : 1,124,563
df_core : 5,622,812


Fetching ... files: 0it [00:00, ?it/s]

Files: 80
Total size: 4850.4 MB
Train pairs: 4,476,056
Core pairs : 5,597,501
RAM available: 30.2 GB


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import gc

TEXT_COLS = ["asin", "reviewerID", "reviewText",
             "text", "title", "brand", "price_clean"]

chunks = []
print("Loading text columns...")

for i, (path, _) in enumerate(downloaded):
    match = re.search(r'overall=(\d)', path)
    overall_val = int(match.group(1)) if match else None

    pf = pq.ParquetFile(path)
    chunk = pf.read(columns=TEXT_COLS).to_pandas()
    chunk[COL_RATING] = overall_val

    pair_series = pd.Series(list(zip(chunk[COL_USER], chunk[COL_ITEM])))
    mask_train = pair_series.isin(train_pairs)

    chunk = chunk.loc[mask_train, [
        COL_ITEM, COL_USER, "reviewText", "text",
        "title", "brand", "price_clean", COL_RATING
    ]]

    if len(chunk) > 0:
        chunks.append(chunk)

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(downloaded)} files")

train_text = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"train_text shape : {train_text.shape}")
print(f"RAM available    : {psutil.virtual_memory().available/1e9:.1f} GB")

Loading text columns...
  20/80 files
  40/80 files
  60/80 files
  80/80 files
train_text shape : (4498974, 8)
RAM available    : 26.1 GB


In [3]:
# Aggregate per item — vectorized groupby
item_reviews = (
    train_text.groupby("asin")["reviewText"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index(name="aggregated_review")
)

item_meta = (
    train_text.drop_duplicates("asin")[["asin", "title", "text"]]
    .set_index("asin")
    .reindex(item_reviews["asin"])
    .reset_index()
)
item_meta["meta_text"] = (
    item_meta["title"].fillna("") + " " +
    item_meta["text"].fillna("")
).str.strip()

item_misc = (
    train_text.groupby("asin")
    .agg(avg_rating=  (COL_RATING, "mean"),
         n_reviews  =  (COL_RATING, "count"),
         price_clean= ("price_clean", "first"),
         brand      = ("brand", "first"))
    .reindex(item_reviews["asin"])
    .reset_index()
)

print(f"Items: {len(item_reviews):,}")
print(f"Avg review text length: "
      f"{item_reviews['aggregated_review'].str.len().mean():.0f} chars")
print(f"Empty reviews: "
      f"{(item_reviews['aggregated_review'].str.strip() == '').sum():,}")
print(f"Empty meta   : "
      f"{(item_meta['meta_text'].str.strip() == '').sum():,}")

Items: 169,241
Avg review text length: 7188 chars
Empty reviews: 0
Empty meta   : 1


In [4]:
# TF-IDF — chỉ chạy sau khi confirm không empty
review_corpus = item_reviews["aggregated_review"].tolist()
meta_corpus   = item_meta["meta_text"].tolist()

tfidf_review = TfidfVectorizer(max_features=3000, max_df=0.95,
                                min_df=5, ngram_range=(1,2),
                                sublinear_tf=True)
tfidf_meta   = TfidfVectorizer(max_features=1000, min_df=3,
                                sublinear_tf=True)

review_matrix = tfidf_review.fit_transform(review_corpus)
meta_matrix   = tfidf_meta.fit_transform(meta_corpus)

print(f"Review TF-IDF : {review_matrix.shape}")
print(f"Meta TF-IDF   : {meta_matrix.shape}")

# Numerical + Brand
price_vals   = item_misc["price_clean"].fillna(
                   item_misc["price_clean"].median()).values
avg_ratings  = item_misc["avg_rating"].fillna(3.5).values
n_reviews    = item_misc["n_reviews"].values

alpha_price  = 1.0 / max(price_vals.mean(), 1e-6)
alpha_rating = 1.0 / max(avg_ratings.mean(), 1e-6)

numerical = csr_matrix(np.column_stack([
    price_vals  * alpha_price,
    avg_ratings * alpha_rating,
    np.log1p(n_reviews),
]))

top_brands   = item_misc["brand"].value_counts().head(100).index
brand_clean  = item_misc["brand"].where(
    item_misc["brand"].isin(top_brands), "other")
brand_dummies = csr_matrix(
    pd.get_dummies(brand_clean, prefix="brand").values.astype(float)
)

item_profiles_raw = hstack([review_matrix, meta_matrix,
                              numerical, brand_dummies]).tocsr()

item_order = item_reviews["asin"].tolist()
item_enc_s2 = {asin: idx for idx, asin in enumerate(item_order)}

del train_text, review_corpus, meta_corpus
gc.collect()

print(f"\nitem_profiles_raw : {item_profiles_raw.shape}")
print(f"RAM available     : {psutil.virtual_memory().available/1e9:.1f} GB")

Review TF-IDF : (169241, 3000)
Meta TF-IDF   : (169241, 1000)

item_profiles_raw : (169241, 4104)
RAM available     : 23.2 GB


In [5]:
# ── TruncatedSVD: 4104 → 128 dims ────────────────────────────
# Đây là fix chính — user_profiles sẽ chỉ tốn 300MB thay vì 9.6GB

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

SVD_DIMS = 128
print(f"Running TruncatedSVD: {item_profiles_raw.shape} → (_, {SVD_DIMS})...")

svd = TruncatedSVD(n_components=SVD_DIMS, random_state=42)
item_profiles = normalize(
    svd.fit_transform(item_profiles_raw).astype(np.float32),
    norm="l2"
)
explained = svd.explained_variance_ratio_.sum()

print(f"item_profiles shape     : {item_profiles.shape}")
print(f"Explained variance      : {explained:.3%}")
print(f"RAM item_profiles       : ~{item_profiles.nbytes/1e9:.3f} GB")

item_enc_s2 = {asin: idx for idx, asin in enumerate(item_order)}

del item_profiles_raw
gc.collect()

ram = psutil.virtual_memory()
print(f"RAM available           : {ram.available/1e9:.1f} GB")

Running TruncatedSVD: (169241, 4104) → (_, 128)...
item_profiles shape     : (169241, 128)
Explained variance      : 74.543%
RAM item_profiles       : ~0.087 GB
RAM available           : 23.9 GB


In [6]:
# ── User Profiles: 586K × 128 = ~300MB ───────────────────────

# User/item encoding cho CF
user_list      = train[COL_USER].unique()
item_list      = train[COL_ITEM].unique()
user_enc_s2    = {u: i for i, u in enumerate(user_list)}
item_enc_s2_cf = {a: i for i, a in enumerate(item_list)}

# Utility matrix
rows = train[COL_USER].map(user_enc_s2).values
cols = train[COL_ITEM].map(item_enc_s2_cf).values
vals = train[COL_RATING].astype(float).values
M    = sp.csr_matrix((vals, (rows, cols)),
                     shape=(len(user_list), len(item_list)))

# Double normalize (vectorized)
M_csr       = M.astype(float)
user_sums   = np.array(M_csr.sum(axis=1)).flatten()
user_counts = np.diff(M_csr.indptr).astype(float)
user_counts[user_counts == 0] = 1
user_means_vec = user_sums / user_counts
row_idx        = np.repeat(np.arange(M_csr.shape[0]), np.diff(M_csr.indptr))
M_csr.data    -= user_means_vec[row_idx]

M_csc       = M_csr.tocsc()
item_sums   = np.array(M_csc.sum(axis=0)).flatten()
item_counts = np.diff(M_csc.indptr).astype(float)
item_counts[item_counts == 0] = 1
item_means_vec = item_sums / item_counts
col_idx        = np.repeat(np.arange(M_csc.shape[1]), np.diff(M_csc.indptr))
M_csc.data    -= item_means_vec[col_idx]
M_norm         = M_csc.tocsr()

print(f"M_norm mean: {M_norm.data.mean():.8f}  (phải ≈ 0)")

# Implicit matrix
C = sp.csr_matrix((np.ones(len(rows)), (rows, cols)),
                  shape=(len(user_list), len(item_list)))

# User profiles — chunked, 128 dims, float32
train_s2             = train.copy()
train_s2["user_idx"] = train_s2[COL_USER].map(user_enc_s2)
train_s2["item_idx"] = train_s2[COL_ITEM].map(item_enc_s2)
train_s2             = train_s2.dropna(subset=["user_idx","item_idx"])
train_s2["user_idx"] = train_s2["user_idx"].astype(int)
train_s2["item_idx"] = train_s2["item_idx"].astype(int)

days_ago         = (train_s2[COL_DATE].max() - train_s2[COL_DATE]).dt.days.values
time_weights     = np.exp(-0.001 * days_ago)
norm_ratings     = (train_s2[COL_RATING].values.astype(float)
                    - user_means_vec[train_s2["user_idx"].values])
combined_weights = (norm_ratings * time_weights).astype(np.float32)

n_users              = len(user_list)
user_profiles_dense  = np.zeros((n_users, SVD_DIMS), dtype=np.float32)
user_idxs            = train_s2["user_idx"].values
item_idxs            = train_s2["item_idx"].values

CHUNK = 500_000
for i in range(0, len(train_s2), CHUNK):
    s  = i
    e  = min(i + CHUNK, len(train_s2))
    np.add.at(user_profiles_dense, user_idxs[s:e],
              item_profiles[item_idxs[s:e]] * combined_weights[s:e, np.newaxis])
    if (i // CHUNK + 1) % 2 == 0:
        print(f"  chunk {i//CHUNK+1}/{len(train_s2)//CHUNK+1}")

norms = np.linalg.norm(user_profiles_dense, axis=1, keepdims=True)
norms[norms == 0] = 1
user_profiles_dense /= norms

print(f"\nUser profiles : {user_profiles_dense.shape}")
print(f"RAM           : ~{user_profiles_dense.nbytes/1e9:.3f} GB")
ram = psutil.virtual_memory()
print(f"RAM available : {ram.available/1e9:.1f} GB")

M_norm mean: -0.00000000  (phải ≈ 0)
  chunk 2/9
  chunk 4/9
  chunk 6/9
  chunk 8/9

User profiles : (620917, 128)
RAM           : ~0.318 GB
RAM available : 22.7 GB


In [7]:
# ── Save tất cả ──────────────────────────────────────────────
import pickle, json
from pathlib import Path

STAGE2_DIR = Path("/kaggle/working/data/stage2_artifacts")
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

sp.save_npz(STAGE2_DIR / "M_norm.npz",     M_norm)
sp.save_npz(STAGE2_DIR / "C_implicit.npz", C)
np.save(STAGE2_DIR / "item_profiles.npy",  item_profiles)
np.save(STAGE2_DIR / "user_profiles.npy",  user_profiles_dense)

with open(STAGE2_DIR / "encodings.pkl", "wb") as f:
    pickle.dump({
        "user_enc_s2"    : user_enc_s2,
        "item_enc_s2_cf" : item_enc_s2_cf,
        "item_enc_s2"    : item_enc_s2,
        "user_list"      : user_list,
        "item_list"      : item_list,
        "user_means_vec" : user_means_vec,
        "item_means_vec" : item_means_vec,
        "global_mean"    : global_mean,
    }, f)

with open(STAGE2_DIR / "tfidf_review.pkl", "wb") as f:
    pickle.dump(tfidf_review, f)
with open(STAGE2_DIR / "svd_model.pkl", "wb") as f:
    pickle.dump(svd, f)

summary = {
    "svd_dims"              : SVD_DIMS,
    "explained_variance"    : float(explained),
    "item_profiles"         : list(item_profiles.shape),
    "user_profiles"         : list(user_profiles_dense.shape),
    "M_norm"                : list(M_norm.shape),
    "lambda_decay"          : 0.001,
}
with open(STAGE2_DIR / "stage2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=== Saved ===")
for f in sorted(STAGE2_DIR.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size/1e6:.1f} MB")
print(f"\nStage 2 complete — {CATEGORY.upper()}")

=== Saved ===
  C_implicit.npz                  13.1 MB
  M_norm.npz                      46.9 MB
  encodings.pkl                   29.5 MB
  item_profiles.npy               86.7 MB
  stage2_summary.json             0.0 MB
  svd_model.pkl                   4.2 MB
  tfidf_review.pkl                0.1 MB
  user_profiles.npy               317.9 MB

Stage 2 complete — HOME_&_KITCHEN


Ý nghĩa của Stage 2
Stage 2 trả lời một câu hỏi cốt lõi: "Biểu diễn users và items dưới dạng số như thế nào để machine có thể tính toán được?"
Raw data chỉ là text và ratings — không thể đưa thẳng vào model. Stage 2 chuyển đổi chúng thành 4 artifacts toán học, mỗi cái capture một góc nhìn khác nhau về data.

4 Artifacts và ý nghĩa
item_profiles.npy — (143,279 × 128)
Mỗi item được biểu diễn bằng một vector 128 chiều. Vector này được học từ TF-IDF của review text, product description, title, brand, và price — sau đó compressed bằng TruncatedSVD. Hai items có vector gần nhau trong không gian 128 chiều là hai items mà users describe bằng ngôn ngữ tương tự, thuộc brand tương tự, price range tương tự. Đây là nền tảng cho content-based filtering.
87.57% explained variance có nghĩa là 128 dims giữ lại gần như toàn bộ thông tin của 4104 dims gốc — compression hiệu quả mà không mất signal.
user_profiles.npy — (586,025 × 128)
Mỗi user được biểu diễn bằng weighted average của các item_profiles mà họ đã rate, với hai điều chỉnh quan trọng. Thứ nhất, normalized rating (trừ user mean) — generous rater cho 5 sao một món đồ tệ sẽ không bị overweight như harsh rater cho 4 sao một món đồ tốt. Thứ hai, time-decay — review 3 năm trước chỉ còn ~33% weight so với review gần đây. User vector capture taste hiện tại của user, không phải taste lịch sử.
Hai users có vector gần nhau = taste tương tự = nền tảng cho user-based collaborative filtering.
M_norm.npz — sparse (586,025 × 143,284)
Utility matrix sau double normalization. Row là users, column là items, value là normalized rating (đã trừ user mean và item mean). Blank = chưa rate, không phải 0.
Double normalization loại bỏ hai loại bias độc lập: user bias (generous vs harsh rater) và item bias (sản phẩm tốt vs tệ). Những gì còn lại trong matrix là preference signal thuần túy — user này thích item này hơn hoặc kém so với expected. Đây là input cho Matrix Factorization (SVD/ALS) và Item-Item CF.
C_implicit.npz — sparse binary (586,025 × 143,284)
Cùng shape với M_norm nhưng mọi value = 1. Không quan tâm rating bao nhiêu sao — chỉ quan tâm "đã mua hay chưa". Capture purchase behavior thay vì opinion. Useful cho ALS implicit vì blank trong matrix này là weak negative signal (chưa mua) chứ không phải unknown.

Mối quan hệ giữa 4 artifacts
item_profiles ──────────────────────► Content-based filtering
      │                                (cosine similarity giữa items)
      │ weighted sum theo ratings
      ▼
user_profiles ──────────────────────► User-based CF
                                       (cosine similarity giữa users)

M_norm ─────────────────────────────► Item-Item CF
      │                                Matrix Factorization (SGD)
      │
C_implicit ─────────────────────────► ALS implicit

Điều quan trọng nhất Stage 2 đã làm đúng
Không có data leakage. Tất cả TF-IDF, SVD, means đều được fit chỉ trên train set. Test set không được "nhìn thấy" trong bất kỳ bước nào. Đây là điều nhiều team hay sai — fit vectorizer trên toàn bộ data rồi mới split.
Stage 2 done. Stage 3 sẽ dùng 4 artifacts này để train các models và so sánh RMSE với baseline 1.2134.